# Project 3: Statistics and Exploratory Data Analysis**Date**: 2026-09-12  **Estimated Time**: 12-15 hours  **Goal**: Understand data through statistics and visualizations before building models

## Learning ObjectivesBy the end of this project, you will:- Master descriptive statistics (mean, median, variance, etc.)- Understand probability distributions- Perform a complete exploratory data analysis- Create meaningful visualizations- Understand correlation vs. causation- Identify data quality issues (missing values, outliers)- Make data-driven decisions

In [ ]:
# Setup: import librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as sns%matplotlib inlinesns.set_style('whitegrid')np.random.seed(42)print('Setup complete!')print(f'NumPy version: {np.__version__}')print(f'Pandas version: {pd.__version__}')

## 1. Problem Definition**Problem**: Perform a complete statistical analysis and EDA on a customer dataset.**Type**: Data analysis and statistics (foundation for ML)**Goal**: Before building models, understand the data by:- Computing descriptive statistics- Visualizing distributions and relationships- Detecting outliers and missing values- Testing simple hypotheses**Why this matters**: ML is only as good as your understanding of the data. Bad data leads to bad models.

## 2. Dataset CreationWe create a realistic customer dataset with intentional issues:- Age, income, spend score, satisfaction- Missing values- Outliers- Correlations between variables

In [ ]:
# Generate synthetic customer datan = 200age = np.random.normal(40, 12, n).astype(int)age = np.clip(age, 18, 75)income = 25000 + age * 800 + np.random.normal(0, 15000, n)income = np.clip(income, 15000, 150000)spend_score = 20 + income / 3000 + np.random.normal(0, 5, n)spend_score = np.clip(spend_score, 0, 100)satisfaction = 3 + (spend_score - 50) / 25 + np.random.normal(0, 0.6, n)satisfaction = np.clip(satisfaction, 1, 5)purchase_freq = np.random.poisson(5, n).astype(int)gender = np.random.choice(['M', 'F'], n)df = pd.DataFrame({    'customer_id': range(1, n + 1),    'age': age,    'income': income.round(2),    'spend_score': spend_score.round(2),    'satisfaction': satisfaction.round(2),    'purchase_freq': purchase_freq,    'gender': gender})# Add missing valuesmissing = np.random.choice(df.index, 10, replace=False)df.loc[missing, 'income'] = np.nan# Add outliersoutliers = np.random.choice(df.index, 5, replace=False)df.loc[outliers, 'income'] = df['income'].max() * 5print(f'Dataset shape: {df.shape}')print('First 5 rows:')print(df.head())

## 3. First InspectionBefore analysis, inspect the data for types, missing values, and basic info.

In [ ]:
print("Data types:")
print(df.dtypes)
print("Missing values per column:")
print(df.isnull().sum())
print("Summary statistics:")
print(df.describe())


## 4. Descriptive Statistics from ScratchImplement statistics manually, then compare with NumPy and Pandas.

In [ ]:
incomes = df["income"].dropna().values


def mean(x):
    return sum(x) / len(x)


def median(x):
    x = sorted(x)
    n = len(x)
    mid = n // 2
    if n % 2 == 0:
        return (x[mid - 1] + x[mid]) / 2
    return x[mid]


def variance(x):
    m = mean(x)
    return sum((xi - m) ** 2 for xi in x) / (len(x) - 1)


def std(x):
    return variance(x) ** 0.5


print("Manual vs. Library:")
print(f"Mean:     {mean(incomes):.2f} | {incomes.mean():.2f}")
print(f"Median:   {median(incomes):.2f} | {np.median(incomes):.2f}")
print(f"Variance: {variance(incomes):.2f} | {incomes.var(ddof=1):.2f}")
print(f"Std:      {std(incomes):.2f} | {incomes.std(ddof=1):.2f}")
print(f"Min/Max:  {min(incomes):.2f} / {max(incomes):.2f}")
print(f"Range:    {max(incomes) - min(incomes):.2f}")


## 5. Percentiles and IQRPercentiles help us understand the spread. The Interquartile Range (IQR) is robust to outliers.

In [ ]:
q1 = np.percentile(incomes, 25)
q2 = np.percentile(incomes, 50)
q3 = np.percentile(incomes, 75)
iqr = q3 - q1print(f"Q1 (25th percentile): {q1:.2f}")
print(f"Q2 (median):          {q2:.2f}")
print(f"Q3 (75th percentile): {q3:.2f}")
print(
    f"IQR:                  {iqr:.2f}"
)  # Manual percentile using linear interpolation


def percentile(x, p):
    x = sorted(x)
    k = (len(x) - 1) * p / 100
    f = int(np.floor(k))
    c = int(np.ceil(k))
    if f == c:
        return x[f]
    return x[f] * (c - k) + x[c] * (k - f)


print(f"Manual Q1: {percentile(incomes, 25):.2f}")
print(f"Manual Q3: {percentile(incomes, 75):.2f}")


## 6. Distribution AnalysisUnderstanding distributions is key. Most ML algorithms assume something about the data distribution.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(df["income"].dropna(), bins=30, edgecolor="black")
plt.title("Income Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.subplot(1, 3, 2)
plt.hist(df["age"], bins=20, edgecolor="black", color="orange")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.subplot(1, 3, 3)
plt.hist(df["spend_score"], bins=20, edgecolor="black", color="green")
plt.title("Spend Score Distribution")
plt.xlabel("Spend Score")
plt.tight_layout()
plt.show()


## 7. Box Plots for Outlier DetectionBox plots show median, quartiles, and outliers.

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(y=df["income"])
plt.title("Income Box Plot")
plt.subplot(1, 2, 2)
sns.boxplot(x=df["gender"], y=df["spend_score"])
plt.title("Spend Score by Gender")
plt.tight_layout()
plt.show()


## 8. Detecting and Handling OutliersUse the IQR method to identify outliers. Decide whether to remove, cap, or keep.

In [ ]:
def detect_outliers_iqr(data):
    q1 = np.percentile(data.dropna(), 25)
    q3 = np.percentile(data.dropna(), 75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (data < lower) | (data > upper)


outliers = detect_outliers_iqr(df["income"])
print(f"Number of income outliers: {outliers.sum()}")
print("Outlier rows:")
print(df[outliers][["customer_id", "income"]])
df_clean = df.copy()
df_clean = df_clean[(~outliers) | (df_clean["income"].isnull())]
print(f"Shape after removing income outliers: {df_clean.shape}")


## 9. Missing Data AnalysisMissing data is common. Understand where it is and how to handle it.

In [ ]:
print("Missing values by column:")
print(df.isnull().sum())
plt.figure(figsize=(8, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Data Map")
plt.show()
median_income = df["income"].median()
df["income_imputed"] = df["income"].fillna(median_income)
print(f"Median income used for imputation: {median_income:.2f}")
print(f'Missing after imputation: {df["income_imputed"].isnull().sum()}')


## 10. Correlation AnalysisCorrelation measures how two variables move together. But correlation is not causation!

In [ ]:
numeric_cols = ["age", "income", "spend_score", "satisfaction", "purchase_freq"]
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, square=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


def pearson(x, y):
    x = x.dropna().values
    y = y.dropna().values
    x_mean = x.mean()
    y_mean = y.mean()
    num = sum((xi - x_mean) * (yi - y_mean) for xi, yi in zip(x, y))
    den = (
        sum((xi - x_mean) ** 2 for xi in x) * sum((yi - y_mean) ** 2 for yi in y)
    ) ** 0.5
    return num / den


r = pearson(df["income"], df["spend_score"])
print(f"Manual correlation (income vs spend_score): {r:.3f}")
income_filled = df["income"].fillna(df["income"].median())
print(f'NumPy correlation: {np.corrcoef(income_filled, df["spend_score"])[0, 1]:.3f}')


## 11. Scatter Plots and RelationshipsVisualize relationships between continuous variables.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.scatter(df["age"], df["income_imputed"], alpha=0.5)
plt.xlabel("Age")
plt.ylabel("Income")
plt.title("Age vs Income")
plt.subplot(1, 3, 2)
plt.scatter(df["income_imputed"], df["spend_score"], alpha=0.5, color="green")
plt.xlabel("Income")
plt.ylabel("Spend Score")
plt.title("Income vs Spend Score")
plt.subplot(1, 3, 3)
plt.scatter(df["spend_score"], df["satisfaction"], alpha=0.5, color="red")
plt.xlabel("Spend Score")
plt.ylabel("Satisfaction")
plt.title("Spend Score vs Satisfaction")
plt.tight_layout()
plt.show()


## 12. Probability and Conditional ProbabilityBasic probability helps us understand how likely events are and how they relate.

In [ ]:
total = len(df)
n_female = (df["gender"] == "F").sum()
n_high = (df["income_imputed"] > 60000).sum()
p_female = n_female / total

p_high = n_high / total

print(f"P(Female):              {p_female:.3f}")
print(f"P(High Income):         {p_high:.3f}")
female_df = df[df["gender"] == "F"]
p_high_given_female = (female_df["income_imputed"] > 60000).mean()
print(f"P(High Income | Female): {p_high_given_female:.3f}")
high_df = df[df["income_imputed"] > 60000]
p_female_given_high = (high_df["gender"] == "F").mean()
print(f"P(Female | High Income): {p_female_given_high:.3f}")


## 13. Hypothesis Testing — Two-Sample t-TestIs there a significant difference in spend score between male and female customers?

In [ ]:
male = df[df["gender"] == "M"]["spend_score"].dropna().values

female = df[df["gender"] == "F"]["spend_score"].dropna().values


def t_test(a, b):
    a, b = np.array(a), np.array(b)
    mean_a, mean_b = a.mean(), b.mean()
    var_a, var_b = a.var(ddof=1), b.var(ddof=1)
    n_a, n_b = len(a), len(b)
    se = np.sqrt(var_a / n_a + var_b / n_b)
    t = (mean_a - mean_b) / se
    deg = n_a + n_b - 2
    return t, deg


t_stat, deg = t_test(male, female)
print(f"Male mean:   {male.mean():.2f}")
print(f"Female mean: {female.mean():.2f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"degrees of freedom: {deg}")
print("A large |t| value suggests a significant difference between the groups.")


## 14. EDA ReportCombine insights into a concise report. Communication is as important as analysis.

In [ ]:
def generate_eda_report(df):
    numeric = df.select_dtypes(include=np.number)
    report = {
        "observations": len(df),
        "features": list(df.columns),
        "missing_total": df.isnull().sum().sum(),
        "missing_by_column": df.isnull().sum().to_dict(),
        "numeric_summary": numeric.describe().to_dict(),
    }
    return report


report = generate_eda_report(df)
print("EDA Report:")
print(f'Observations: {report["observations"]}')
print(f'Features: {report["features"]}')
print(f'Total missing: {report["missing_total"]}')
print("Top recommendations:")
print("1. Handle missing income values before modeling.")
print("2. Investigate extreme income outliers.")
print("3. Spend score and income are correlated — use both carefully.")
print("4. No strong gender difference in spend score based on t-test.")


## 15. Exercises### Beginner1. Compute the mode of purchase_freq manually.2. Create a histogram of satisfaction with 10 bins.3. Calculate the variance of spend_score using NumPy and manually.### Intermediate1. Perform an EDA by grouping customers into age brackets.2. Detect outliers in spend_score using the z-score method.3. Compute the correlation between age and satisfaction.### Advanced1. Implement a chi-square test for independence between gender and high income.2. Build a function that automatically generates an EDA report for any DataFrame.3. Compare mean income between two groups using a bootstrap confidence interval.

## 16. Key TakeawaysAfter completing this project, you should understand:- Descriptive statistics and what they tell you- How to detect and handle outliers- How to handle missing data- Correlation vs. causation- Basic probability and hypothesis testing- How to create an EDA report### Next Project**Project 4: Data Visualization** — mastering Matplotlib and Seaborn for ML.